In [0]:
# Cell 1: setup
from pyspark.sql import functions as F
from pyspark.sql.functions import col

CATALOG = "city_lens"
spark.conf.set("spark.sql.session.timeZone", "Europe/Amsterdam")

h = spark.table(f"{CATALOG}.gold.happenings_enriched")

In [0]:
# Cell 2: gold.daily_activity. One row per city per day
days = spark.sql("""
    SELECT explode(sequence(date_sub(current_date(), 30), date_add(current_date(), 30))) AS day
""")
cities = h.select("city_key").distinct()
spine = cities.crossJoin(days)                      # every city x every day

# Roadworks active on each day (scheduled rows overlap the day)
works = (
    spine.join(h.where(col("time_basis") == "scheduled"),
               ["city_key"])
         .where(col("day").between(F.to_date("start_local"), F.to_date("end_local")))
         .groupBy("city_key", "day")
         .agg(F.count("*").alias("active_works"),
              F.sum(F.when(col("closure_level") == "road_closed", 1).otherwise(0)).alias("active_closures"),
              F.sum(F.when(col("category") == "road_event", 1).otherwise(0)).alias("road_events"))
)

# Notices published on each day, one column per category
notices = (
    h.where(col("time_basis") == "published")
     .groupBy("city_key", col("published_date").alias("day"))
     .pivot("category", ["building_works", "street_works", "tree_felling",
                         "event_permit", "traffic_measure", "other"])
     .count()
)

weather = (
    spark.table(f"{CATALOG}.silver.weather_hourly")
    .where(F.hour("hour_local").between(7, 22))
    .groupBy("city_key", col("date_local").alias("day"))
    .agg(F.max("rain_chance_pct").alias("max_rain_pct"),
         F.round(F.max("temperature_c"), 1).alias("max_temp_c"))
)

daily = (
    spine.join(works, ["city_key", "day"], "left")
         .join(notices, ["city_key", "day"], "left")
         .join(weather, ["city_key", "day"], "left")
         .fillna(0, subset=["active_works", "active_closures", "road_events",
                            "building_works", "street_works", "tree_felling",
                            "event_permit", "traffic_measure", "other"])
         .withColumn("new_notices", col("building_works") + col("street_works") + col("tree_felling")
                                    + col("event_permit") + col("traffic_measure") + col("other"))
         .withColumn("weekday", F.date_format("day", "EEEE"))
         .withColumn("refreshed_at", F.current_timestamp())
)

(daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.gold.daily_activity"))

display(spark.table(f"{CATALOG}.gold.daily_activity")
        .where(col("day").between(F.date_sub(F.current_date(), 3), F.date_add(F.current_date(), 3)))
        .orderBy("day"))

In [0]:
# Cell 3: gold.permits_weekly. New notices per week, category and stage
weekly = (
    h.where(col("time_basis") == "published")
     .withColumn("week_start", F.to_date(F.date_trunc("week", "published_date")))   # Monday
     .groupBy("city_key", "week_start", "category", "stage")
     .agg(F.count("*").alias("notices"),
          F.sum(F.when(col("badge_closure_nearby"), 1).otherwise(0)).alias("near_a_closure"))
     .withColumn("refreshed_at", F.current_timestamp())
)

(weekly.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.gold.permits_weekly"))

display(spark.sql(f"""
    SELECT week_start, sum(notices) AS notices
    FROM {CATALOG}.gold.permits_weekly
    GROUP BY week_start
    ORDER BY week_start
"""))